# SALES PREDICTION USING PYTHON

This notebook explores advertising data and trains a Linear Regression model to estimate sales. Run the cells from top to bottom.

The CSV must be at `data/advertising.csv` and contain numeric columns `TV`, `Radio`, `Newspaper`, and `Sales`. The first three are input features; `Sales` is the target.

In [ ]:
from pathlib import Path
import json
import math

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

project_dir = Path.cwd()
if not (project_dir / 'data' / 'advertising.csv').exists():
    if project_dir.name == 'notebooks':
        project_dir = project_dir.parent
    elif (project_dir / 'Sales_Prediction' / 'data' / 'advertising.csv').exists():
        project_dir = project_dir / 'Sales_Prediction'
data_path = project_dir / 'data' / 'advertising.csv'
output_dir = project_dir / 'outputs'
model_dir = project_dir / 'models'
output_dir.mkdir(parents=True, exist_ok=True)
model_dir.mkdir(parents=True, exist_ok=True)
print(f'Project folder: {project_dir}')

## Load and validate the data

The shared public CSV sometimes contains an extra unnamed row-number column. It is removed because it is not a model feature. Missing, non-numeric, infinite, or negative advertising values stop the workflow so they can be corrected.

In [ ]:
if not data_path.exists():
    raise FileNotFoundError(f'Dataset not found: {data_path}. Place advertising.csv in data/.')

df = pd.read_csv(data_path)
df = df.loc[:, ~df.columns.astype(str).str.match(r'^Unnamed:')]
required_columns = ['TV', 'Radio', 'Newspaper', 'Sales']
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f'Missing required CSV columns: {missing_columns}. Expected: {required_columns}')
df = df[required_columns].copy()
for column in required_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')
if df.empty:
    raise ValueError('The CSV has no data rows.')
if df.isna().any().any():
    raise ValueError(f'Missing or non-numeric values by column: {df.isna().sum().to_dict()}')
if not np.isfinite(df.to_numpy()).all():
    raise ValueError('The data contains infinite values; replace them with finite numbers.')
if (df[['TV', 'Radio', 'Newspaper']] < 0).any().any():
    raise ValueError('Advertising budgets cannot be negative.')
print('First five rows:')
display(df.head())
print(f'Rows and columns: {df.shape}')
print('Column names:', df.columns.tolist())
print('Data types:')
display(df.dtypes)
print('Missing values:')
display(df.isna().sum())
print('Duplicate rows:', df.duplicated().sum())
print('Basic statistics:')
display(df.describe())
print('Numerical correlations:')
display(df.corr(numeric_only=True))

## Explore the relationships

The three scatter plots show each advertising channel against sales. The heatmap summarizes pairwise numerical correlation, and the histogram shows the sales distribution. The PNGs are saved in `outputs/`.

In [ ]:
sns.set_theme(style='whitegrid')
features = ['TV', 'Radio', 'Newspaper']
figure, axes = plt.subplots(1, 3, figsize=(16, 5))
for axis, feature in zip(axes, features):
    sns.scatterplot(data=df, x=feature, y='Sales', ax=axis, alpha=0.8)
    axis.set_title(f'{feature} Advertising vs Sales')
    axis.set_xlabel(f'{feature} advertising budget')
    axis.set_ylabel('Sales')
figure.tight_layout()
figure.savefig(output_dir / 'advertising_vs_sales.png', dpi=150)
plt.show()

figure, axis = plt.subplots(figsize=(8, 6))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm', center=0, fmt='.2f', ax=axis)
axis.set_title('Correlation Between Advertising and Sales')
figure.tight_layout()
figure.savefig(output_dir / 'correlation_heatmap.png', dpi=150)
plt.show()

figure, axis = plt.subplots(figsize=(8, 5))
sns.histplot(data=df, x='Sales', kde=True, ax=axis)
axis.set_title('Distribution of Sales')
axis.set_xlabel('Sales')
axis.set_ylabel('Number of records')
figure.tight_layout()
figure.savefig(output_dir / 'sales_distribution.png', dpi=150)
plt.show()

## Separate inputs and target, then train

`X` contains the information available to make a prediction. `y` contains the outcome the model should learn. We hold back 20% of the rows to test predictions on data that was not used for fitting. The fixed random state makes the split repeatable.

In [ ]:
X = df[['TV', 'Radio', 'Newspaper']]
y = df['Sales']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
model = LinearRegression()
model.fit(X_train, y_train)
training_predictions = model.predict(X_train)
test_predictions = model.predict(X_test)
print(f'Intercept: {model.intercept_:.4f}')
print('Coefficients:')
for feature, coefficient in zip(X.columns, model.coef_):
    print(f'  {feature}: {coefficient:.4f}')
print('Each coefficient estimates the change in sales for one budget-unit increase, holding the other budgets constant.')
print('First 10 training predictions:')
display(pd.DataFrame({'Actual Sales': y_train, 'Predicted Sales': training_predictions}).head(10))
print('Testing predictions:')
display(pd.DataFrame({'Actual Sales': y_test, 'Predicted Sales': test_predictions}))

## Evaluate and save the model

MAE is average absolute error, MSE averages squared errors, and RMSE is in sales units. RÂ² measures variance explained on this test split. A high RÂ² is not a guarantee of perfect predictions. The charts compare test predictions with actual values and inspect residual errors.

In [ ]:
metrics = {
    'mae': float(mean_absolute_error(y_test, test_predictions)),
    'mse': float(mean_squared_error(y_test, test_predictions)),
    'rmse': float(mean_squared_error(y_test, test_predictions) ** 0.5),
    'r2': float(r2_score(y_test, test_predictions)),
}
print(f"MAE:  {metrics['mae']:.4f}")
print(f"MSE:  {metrics['mse']:.4f}")
print(f"RMSE: {metrics['rmse']:.4f}")
print(f"RÂ²:   {metrics['r2']:.4f}")

figure, axis = plt.subplots(figsize=(8, 6))
axis.scatter(y_test, test_predictions, alpha=0.8, label='Test observations')
lower = min(float(y_test.min()), float(test_predictions.min()))
upper = max(float(y_test.max()), float(test_predictions.max()))
axis.plot([lower, upper], [lower, upper], color='red', linestyle='--', label='Perfect prediction')
axis.set_title('Actual Sales vs Predicted Sales')
axis.set_xlabel('Actual Sales')
axis.set_ylabel('Predicted Sales')
axis.legend()
figure.tight_layout()
figure.savefig(output_dir / 'actual_vs_predicted.png', dpi=150)
plt.show()

residuals = y_test - test_predictions
figure, axis = plt.subplots(figsize=(8, 5))
axis.scatter(test_predictions, residuals, alpha=0.8)
axis.axhline(0, color='red', linestyle='--', label='Zero error')
axis.set_title('Prediction Residuals')
axis.set_xlabel('Predicted Sales')
axis.set_ylabel('Actual Sales - Predicted Sales')
axis.legend()
figure.tight_layout()
figure.savefig(output_dir / 'residuals.png', dpi=150)
plt.show()

joblib.dump(model, model_dir / 'sales_prediction_model.pkl')
(model_dir / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print(f'Model saved in {model_dir}')
print(f'Plots saved in {output_dir}')

## Predict sales for new budgets

This example predicts for TV=150, Radio=25, and Newspaper=20. The prediction is calculated from the fitted model, not entered as a fixed answer. User-entered values should be finite, numeric, and non-negative.

In [ ]:
def predict_sales(tv, radio, newspaper):
    values = [tv, radio, newspaper]
    if any(not isinstance(value, (int, float)) or not math.isfinite(value) for value in values):
        raise ValueError('Enter finite numeric budgets.')
    if any(value < 0 for value in values):
        raise ValueError('Advertising budgets cannot be negative.')
    input_row = pd.DataFrame([values], columns=['TV', 'Radio', 'Newspaper'])
    return float(model.predict(input_row)[0])

example_prediction = predict_sales(150, 25, 20)
print(f'Predicted Sales: {example_prediction:.2f}')